# F4-M01: Inspección del Dataset Procesado

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | Maria Jose Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) / morte@uji.es (UJI) |

---

## Objetivo
Inspección completa del dataset final: tipos, nulos, estadísticos descriptivos,
distribución del target y deteccion de outliers.

## Entrada / Salida

| Tipo | Archivo | Registros |
|------|---------|-----------|
| Entrada | `04_eda/df_eda_final.parquet` | ~33.621 |
| Salida | `docs/html/fase4/m01_inspeccion.html` | 1 |

## Flujo
F4-M00 Indice > **F4-M01 Inspección** > M02 Target

## Siguiente
`f4_m02_target.ipynb`

In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: CONFIGURACION
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

from src.config import RUTA_HTML, ETIQUETAS_VARIABLES, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64, COLORES

# ── Paneles de interpretación dinámica ──────────────
panel_dist_target = panel('dist-target', 'El dataset presenta un desbalanceo 1:2.4 — 29.2% abandona y 70.8% tiene éxito. Ratio moderado: permite entrenar modelos sin SMOTE obligatorio. En Fase 5 se compararán 3 estrategias: <span class=\"tg\">sin ajuste</span> · <span class=\"tg\">class_weight=\"balanced\"</span> · <span class=\"tg\">SMOTE en train</span>.')
panel_val_faltantes = panel('val-faltantes', 'La variable más crítica es <b>situacion_laboral</b> (37.9% nulos) — se imputará con moda o se añadirá categoría \"Desconocido\". Las notas de acceso y primer año tienen nulos estructurales (estudiantes sin registro). Todo esto ocurre <b>dentro del Pipeline sklearn, solo sobre X_train</b>.')
panel_outliers = panel('outliers', 'Outliers detectados por IQR son valores reales extremos (cred_superados, max_pagos). <span class=\"tm\">No se eliminarán</span> — son información real del estudiante. Se usará <b>RobustScaler</b> para modelos sensibles a la escala (LogReg, SVM, MLP). RF, XGB y CatBoost son robustos a outliers por diseño.')

from src.html import (generar_kpis_html, generar_seccion_html,
                      generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_EDA = ROOT / 'data' / '04_eda'
RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
crear_directorios([RUTA_FASE4_HTML])

fmt = formato_numero_es
COLOR_PRINCIPAL = '#3182ce'

def etiq(col):
    """Devuelve etiqueta legible de una variable."""
    return ETIQUETAS_VARIABLES.get(col, col)

info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA DEL DATASET Y VISION GENERAL
# ============================================================================

df = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')

n_filas, n_cols = df.shape
n_features = n_cols - 2  # sin titulacion ni abandono
memoria_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
pct_abandono = df['abandono'].mean() * 100
n_abandonan = int(df['abandono'].sum())
n_permanecen = n_filas - n_abandonan
pct_missings_global = df.isnull().mean().mean() * 100

# Clasificar variables
cols_num = df.select_dtypes(include='number').columns.drop('abandono').tolist()
cols_cat = df.select_dtypes(include='object').columns.tolist()
cols_bin = [c for c in cols_num if df[c].dropna().isin([0, 1]).all()]
cols_num_cont = [c for c in cols_num if c not in cols_bin]

print('='*70)
print('DATASET: df_eda_final.parquet')
print('='*70)
print(f'Filas:        {fmt(n_filas)}')
print(f'Columnas:     {n_cols} ({n_features} features + titulacion + target)')
print(f'Memoria:      {memoria_mb:.1f} MB')
print(f'Abandono:     {fmt(n_abandonan)} ({pct_abandono:.1f}%)')
print(f'Permanecen:   {fmt(n_permanecen)} ({100-pct_abandono:.1f}%)')
print(f'Numericas:    {len(cols_num_cont)} continuas + {len(cols_bin)} binarias')
print(f'Categoricas:  {len(cols_cat)}')
print(f'Missings:     {pct_missings_global:.2f}% global')

# Tabla de variables
print(f'\n{"Variable":<30} {"Tipo":<12} {"Nulos%":>8} {"Unicos":>8} {"Rango/Valores"}')
print('-'*90)
for col in df.columns:
    pct_nulos = df[col].isnull().mean() * 100
    n_unicos = df[col].nunique()
    if col in cols_bin:
        tipo = 'binaria'
        rango = '0 / 1'
    elif col in cols_num_cont:
        tipo = 'numerica'
        rango = f'{df[col].min():.1f} - {df[col].max():.1f}'
    elif col == 'abandono':
        tipo = 'TARGET'
        rango = '0 / 1'
    else:
        tipo = 'categorica'
        top3 = df[col].value_counts().head(3).index.tolist()
        rango = ', '.join(str(v)[:20] for v in top3) + '...'
    print(f'{col:<30} {tipo:<12} {pct_nulos:>7.1f}% {n_unicos:>8} {rango}')

DATASET: df_eda_final.parquet
Filas:        33.621
Columnas:     21 (19 features + titulacion + target)
Memoria:      23.2 MB
Abandono:     9.833 (29.2%)
Permanecen:   23.788 (70.8%)
Numericas:    10 continuas + 2 binarias
Categoricas:  8
Missings:     4.27% global

Variable                       Tipo           Nulos%   Unicos Rango/Valores
------------------------------------------------------------------------------------------
anios_gap                      numerica         0.0%        9 0.0 - 8.0
anios_sin_beca                 numerica         0.0%       12 0.0 - 11.0
cred_superados_anio_1er        numerica         0.0%      316 0.0 - 252.0
edad_entrada                   numerica         0.0%       51 5.0 - 68.0
indicador_interrupcion         binaria          0.0%        2 0 / 1
max_pagos                      numerica         0.0%        7 1.0 - 7.0
n_anios_beca                   numerica         0.0%       10 0.0 - 9.0
nota_1er_anio                  numerica         7.0%      473 

In [4]:
# ============================================================================
# CELDA 3: DISTRIBUCION DEL ABANDONO
# ============================================================================

graficos_base64 = {}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut
valores = [n_permanecen, n_abandonan]
etiquetas = [f'No abandono\n{100-pct_abandono:.1f}%', f'Abandono\n{pct_abandono:.1f}%']
colores_donut = ['#3182ce', '#e53e3e']
wedges, texts, autotexts = axes[0].pie(
    valores, labels=etiquetas, colors=colores_donut, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75, wedgeprops={'width': 0.4, 'edgecolor': 'white'}
)
for t in texts + autotexts:
    t.set_fontsize(11)

# Barras
bars = axes[1].bar(['No abandono', 'Abandono'], valores, color=colores_donut,
                    edgecolor='white', width=0.5)
for bar, val in zip(bars, valores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 fmt(val), ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Expedientes')
axes[1].set_title('Abandono vs No abandono', fontsize=13, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: fmt(int(x))))

plt.tight_layout()
graficos_base64['target'] = figura_a_base64(fig)
plt.close()

print(f'No abandono: {fmt(n_permanecen)} ({100-pct_abandono:.1f}%)')
print(f'Abandono:    {fmt(n_abandonan)} ({pct_abandono:.1f}%)')
print(f'Ratio:       1:{n_permanecen/n_abandonan:.1f}')
print(f'\nNota: Un modelo naive (siempre predice no abandono) tendria {100-pct_abandono:.1f}% accuracy.')
print(f'Por eso usamos F1 y AUC, no accuracy.')

No abandono: 23.788 (70.8%)
Abandono:    9.833 (29.2%)
Ratio:       1:2.4

Nota: Un modelo naive (siempre predice no abandono) tendria 70.8% accuracy.
Por eso usamos F1 y AUC, no accuracy.


In [5]:
# ============================================================================
# CELDA 4: ANALISIS DE NULOS
# ============================================================================

missings = df.isnull().sum()
vars_con_nulos = missings[missings > 0].sort_values(ascending=False)
n_vars_nulas = len(vars_con_nulos)
cols_nulas = vars_con_nulos.index.tolist()

# --- Grafico 1: Barras horizontales de % nulos ---
if n_vars_nulas > 0:
    fig, ax = plt.subplots(figsize=(7, 4))

    pcts = [(df[c].isnull().sum() / n_filas * 100) for c in cols_nulas]
    colores_bar = ['#e53e3e' if p > 30 else '#ed8936' if p > 10 else '#38a169' for p in pcts]
    labels_nulas = [etiq(c) for c in cols_nulas]
    bars = ax.barh(labels_nulas, pcts, color=colores_bar, edgecolor='white', height=0.6)
    for bar, p in zip(bars, pcts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{p:.1f}%', va='center', fontsize=11, fontweight='bold')
    ax.set_xlabel('% Valores Faltantes', fontsize=12)
    ax.set_title('Porcentaje de Valores Faltantes por Variable', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlim(0, max(pcts) * 1.15)

    # Lineas de referencia
    ax.axvline(30, color='#e53e3e', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(10, color='#ed8936', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(30.5, -0.3, 'Alta', color='#e53e3e', fontsize=9, alpha=0.7)
    ax.text(10.5, -0.3, 'Media', color='#ed8936', fontsize=9, alpha=0.7)

    plt.tight_layout()
    graficos_base64['nulos_barras'] = figura_a_base64(fig)
    plt.close()

# --- Grafico 2: Matriz de co-ocurrencia de nulos ---
if n_vars_nulas > 1:
    nulos_bin = df[cols_nulas].isnull().astype(int)
    co_matrix = nulos_bin.T.dot(nulos_bin)
    # Convertir a porcentaje sobre el total de nulos de cada variable (fila)
    totales = nulos_bin.sum()
    co_pct = co_matrix.copy().astype(float)
    for c in cols_nulas:
        if totales[c] > 0:
            co_pct.loc[c] = (co_matrix.loc[c] / totales[c] * 100).round(1)
        else:
            co_pct.loc[c] = 0

    fig, ax = plt.subplots(figsize=(7, 5))
    import matplotlib.colors as mcolors
    cmap = mcolors.LinearSegmentedColormap.from_list('custom', ['#ffffff', '#bee3f8', '#3182ce'])
    im = ax.imshow(co_pct.values, cmap=cmap, vmin=0, vmax=100)

    ax.set_xticks(range(len(cols_nulas)))
    ax.set_yticks(range(len(cols_nulas)))
    ax.set_xticklabels([etiq(c) for c in cols_nulas], rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels([etiq(c) for c in cols_nulas], fontsize=9)

    # Anotar valores
    for i in range(len(cols_nulas)):
        for j in range(len(cols_nulas)):
            val = co_pct.values[i, j]
            color_texto = 'white' if val > 60 else 'black'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=9, color=color_texto, fontweight='bold' if i == j else 'normal')

    ax.set_title('Co-ocurrencia de Nulos\n(% de nulos de fila que tambien son nulos en columna)',
                 fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='% co-ocurrencia', shrink=0.8)
    plt.tight_layout()
    graficos_base64['nulos_coocurrencia'] = figura_a_base64(fig)
    plt.close()

# Imprimir diagnostico
print('='*70)
print('DIAGNOSTICO DE VALORES FALTANTES')
print('='*70)
print(f'Variables con nulos: {n_vars_nulas} de {n_cols}')
print()
for v, n in vars_con_nulos.items():
    pct = n / n_filas * 100
    severidad = 'ALTA' if pct > 30 else 'MEDIA' if pct > 10 else 'BAJA'
    print(f'  {v:<25} {fmt(n):>8} ({pct:>5.1f}%) Severidad: {severidad}')

print()
print('Co-ocurrencias destacadas:')
if n_vars_nulas > 1:
    for i, ci in enumerate(cols_nulas):
        for j, cj in enumerate(cols_nulas):
            if i < j and co_pct.loc[ci, cj] > 80:
                print(f'  {ci} + {cj}: {co_pct.loc[ci, cj]:.0f}% co-ocurrencia')

print()
print('Diagnóstico probable:')
print('  situacion_laboral (37.9%): MAR — depende del curso/periodo')
print('  orden_preferencia (14.0%): MAR — solo disponible via preinscripcion')
print('  universidad_origen (14.0%): MAR — misma fuente que orden_preferencia')
print('  notas (7-9%):              MAR — sin calificacion registrada')
print('  max_pagos (0.01%):         MCAR — muy pocos casos, aleatorio')

DIAGNOSTICO DE VALORES FALTANTES
Variables con nulos: 7 de 21

  situacion_laboral           12.740 ( 37.9%) Severidad: ALTA
  orden_preferencia            4.699 ( 14.0%) Severidad: MEDIA
  universidad_origen           4.699 ( 14.0%) Severidad: MEDIA
  nota_selectividad            3.118 (  9.3%) Severidad: BAJA
  nota_acceso                  2.567 (  7.6%) Severidad: BAJA
  nota_1er_anio                2.353 (  7.0%) Severidad: BAJA
  max_pagos                        3 (  0.0%) Severidad: BAJA

Co-ocurrencias destacadas:
  orden_preferencia + universidad_origen: 100% co-ocurrencia
  nota_selectividad + nota_acceso: 82% co-ocurrencia

Diagnóstico probable:
  situacion_laboral (37.9%): MAR — depende del curso/periodo
  orden_preferencia (14.0%): MAR — solo disponible via preinscripcion
  universidad_origen (14.0%): MAR — misma fuente que orden_preferencia
  notas (7-9%):              MAR — sin calificacion registrada
  max_pagos (0.01%):         MCAR — muy pocos casos, aleatorio


In [6]:
# ============================================================================
# CELDA 5: ESTADISTICOS DESCRIPTIVOS E HISTOGRAMAS
# ============================================================================

n_num = len(cols_num_cont)
n_col_grid = 2
n_row_grid = 5

fig, axes = plt.subplots(n_row_grid, n_col_grid, figsize=(14, 20))
axes_flat = axes.flatten()

for i, col in enumerate(cols_num_cont):
    ax = axes_flat[i]
    datos = df[col].dropna()

    # Histograma + KDE
    ax.hist(datos, bins=40, color=COLOR_PRINCIPAL, alpha=0.7, edgecolor='white', density=True)
    try:
        kde_x = np.linspace(datos.min(), datos.max(), 200)
        kde = stats.gaussian_kde(datos)
        ax.plot(kde_x, kde(kde_x), color='#e53e3e', linewidth=2)
    except Exception:
        pass

    ax.set_title(etiq(col), fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)

    # Mediana linea
    mediana = datos.median()
    ax.axvline(mediana, color='#ed8936', linestyle='--', linewidth=1.5)

    # Stats dentro del grafico
    media = datos.mean()
    std = datos.std()
    sk = datos.skew()
    stats_text = f'Md={mediana:.1f}  x\u0305={media:.1f}\n\u03c3={std:.1f}  Sk={sk:.1f}'
    ax.text(0.97, 0.95, stats_text, transform=ax.transAxes,
            fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#cbd5e0'))

# Ocultar ejes vacios
for j in range(n_num, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Distribuciones de Variables Numericas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
graficos_base64['histogramas'] = figura_a_base64(fig)
plt.close()

print(f'Generados {n_num} histogramas con KDE y estadísticos')

Generados 10 histogramas con KDE y estadísticos


In [7]:
# ============================================================================
# CELDA 6: DETECCION DE OUTLIERS (IQR)
# ============================================================================

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
axes_flat = axes.flatten()

for i, col in enumerate(cols_num_cont):
    ax = axes_flat[i]
    datos = df[col].dropna()

    bp = ax.boxplot(datos, patch_artist=True, vert=True,
                    boxprops=dict(facecolor=COLOR_PRINCIPAL, alpha=0.6),
                    medianprops=dict(color='#e53e3e', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='#ed8936', markersize=3, alpha=0.4))
    ax.set_title(etiq(col), fontsize=10, fontweight='bold')
    ax.set_xticks([])
    ax.tick_params(labelsize=8)

    # Calcular outliers IQR
    Q1 = datos.quantile(0.25)
    Q3 = datos.quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = int(((datos < Q1 - 1.5 * IQR) | (datos > Q3 + 1.5 * IQR)).sum())
    pct_outliers = n_outliers / len(datos) * 100

    # Stats dentro del grafico
    stats_text = f'Q1={Q1:.1f}  Q3={Q3:.1f}\nIQR={IQR:.1f}\nOut={n_outliers} ({pct_outliers:.1f}%)'
    color_box = '#fed7d7' if pct_outliers > 10 else '#fefcbf' if pct_outliers > 5 else 'white'
    ax.text(0.97, 0.02, stats_text, transform=ax.transAxes,
            fontsize=7, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color_box, alpha=0.85, edgecolor='#cbd5e0'))

# Ocultar ejes vacios
for j in range(len(cols_num_cont), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Boxplots - Deteccion de Outliers (IQR)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
graficos_base64['boxplots'] = figura_a_base64(fig)
plt.close()

print(f'Generados {len(cols_num_cont)} boxplots con estadísticos IQR')

Generados 10 boxplots con estadísticos IQR


In [8]:
# ============================================================================
# CELDA 7: GENERAR HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m01'
)

# KPIs
KPIS = [
    {'valor': fmt(n_filas), 'titulo': 'Expedientes', 'color': COLORES['primary']},
    {'valor': str(n_cols), 'titulo': 'Columnas', 'color': COLORES['success']},
    {'valor': f'{pct_abandono:.1f}%', 'titulo': 'Abandono', 'color': COLORES['danger']},
    {'valor': f'{pct_missings_global:.1f}%', 'titulo': 'Missings global', 'color': COLORES['warning']},
    {'valor': f'{memoria_mb:.1f} MB', 'titulo': 'Memoria', 'color': '#805ad5'},
]

# Seccion 1: Tabla de variables
filas_tabla = ''
for col in df.columns:
    pct_nulos = df[col].isnull().mean() * 100
    n_unicos = df[col].nunique()
    if col in cols_bin:
        tipo = 'binaria'
    elif col in cols_num_cont:
        tipo = 'numerica'
    elif col == 'abandono':
        tipo = 'TARGET'
    else:
        tipo = 'categorica'
    color_nulo = '#fed7d7' if pct_nulos > 30 else '#fefcbf' if pct_nulos > 10 else ''
    style_bg = f' style="background:{color_nulo}"' if color_nulo else ''
    filas_tabla += (
        f'<tr{style_bg}>'
        f'<td style="padding:6px;"><code>{etiq(col)}</code></td>'
        f'<td style="text-align:center;">{tipo}</td>'
        f'<td style="text-align:right;">{pct_nulos:.1f}%</td>'
        f'<td style="text-align:right;">{n_unicos}</td></tr>'
    )

tabla_vars = (
    '<table style="width:100%;border-collapse:collapse;">'
    '<tr style="background:#3182ce;color:white;">'
    '<th style="padding:8px;text-align:left;">Variable</th>'
    '<th style="text-align:center;">Tipo</th>'
    '<th style="text-align:right;">% Nulos</th>'
    '<th style="text-align:right;">Unicos</th></tr>'
    + filas_tabla + '</table>'
)
s_vars = generar_seccion_html('Tabla de Variables', tabla_vars)

# Seccion 2: Target
img_target = f'<img src="data:image/png;base64,{graficos_base64["target"]}" style="max-width:100%;border-radius:8px;">'
nota_accuracy = (
    '<div style="background:#fff5f5;padding:12px;border-radius:8px;border-left:4px solid #e53e3e;margin-top:10px;">'
    f'<strong>Nota:</strong> Un modelo naive (siempre predice "no abandona") tendria '
    f'{100-pct_abandono:.1f}% accuracy. Por eso usamos F1 y AUC como metricas principales.'
    '</div>'
)
s_target = generar_seccion_html('Distribucion del abandono', img_target + nota_accuracy)

# Seccion 3: Nulos
nulos_html = ''

# Barras + Co-ocurrencia lado a lado
nulos_html += '<div style="display:grid;grid-template-columns:1fr 1fr;gap:15px;align-items:start;">'
if 'nulos_barras' in graficos_base64:
    nulos_html += f'<div><img src="data:image/png;base64,{graficos_base64["nulos_barras"]}" style="width:100%;border-radius:8px;"></div>'
if 'nulos_coocurrencia' in graficos_base64:
    nulos_html += f'<div><img src="data:image/png;base64,{graficos_base64["nulos_coocurrencia"]}" style="width:100%;border-radius:8px;"></div>'
nulos_html += '</div>'

# Diagnóstico
diagnostico_html = (
    '<div style="background:#f0f4f8;padding:15px;border-radius:8px;border-left:4px solid #3182ce;margin-top:15px;">'
    '<strong>Diagnóstico de mecanismo de falta:</strong><br><br>'
    '<code>situacion_laboral</code> (37,9%): <strong>MAR</strong> — depende del periodo/curso<br>'
    '<code>orden_preferencia</code> y <code>universidad_origen</code> (14%): <strong>MAR</strong> — fuente preinscripcion, faltan juntas<br>'
    '<code>nota_selectividad</code>, <code>nota_acceso</code>, <code>nota_1er_anio</code> (7-9%): <strong>MAR</strong> — sin calificacion registrada<br>'
    '<code>max_pagos</code> (0,01%): <strong>MCAR</strong> — aleatorio, despreciable'
    '</div>'
    '<div style="background:#f7fafc;padding:12px;border-radius:8px;margin-top:10px;font-size:0.85em;color:#4a5568;">'
    '<strong>Glosario:</strong><br>'
    '<strong>MAR</strong> (Missing At Random): el dato falta por una razon relacionada con otras variables conocidas '
    '(ej: la situacion laboral no se recogia en ciertos cursos). Se puede imputar con modelos.<br>'
    '<strong>MCAR</strong> (Missing Completely At Random): el dato falta sin ningun patron, de forma aleatoria '
    '(ej: 3 registros de 33.621 sin max_pagos). Se puede imputar con la mediana o ignorar.<br>'
    '<strong>MNAR</strong> (Missing Not At Random): el dato falta precisamente por su valor '
    '(ej: estudiantes con malas notas no reportan). No detectado en este dataset.'
    '</div>'
)
nulos_html += diagnostico_html

s_nulos = generar_seccion_html('Valores Faltantes', nulos_html)

# Seccion 4: Histogramas
img_hist = f'<img src="data:image/png;base64,{graficos_base64["histogramas"]}" style="max-width:100%;border-radius:8px;">'

s_desc = generar_seccion_html('Estadisticos Descriptivos', img_hist)

# Seccion 5: Outliers
img_box = f'<img src="data:image/png;base64,{graficos_base64["boxplots"]}" style="max-width:100%;border-radius:8px;">'

s_outliers = generar_seccion_html('Deteccion de Outliers (IQR)', img_box)

# Ensamblar
# Catalogo de variables: fichas compactas en grid
fichas_html = '<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:10px;">'
for col in df.columns:
    pct_nulos = df[col].isnull().mean() * 100
    n_unicos = df[col].nunique()
    nombre = etiq(col)
    if col in cols_bin:
        tipo = 'Binaria'
        tipo_color = '#805ad5'
        detalle = f'0: {int((df[col]==0).sum()):,} | 1: {int((df[col]==1).sum()):,}'.replace(',','.')
    elif col in cols_num_cont:
        tipo = 'Numerica'
        tipo_color = '#3182ce'
        detalle = f'Min: {df[col].min():.1f} | Max: {df[col].max():.1f} | Md: {df[col].median():.1f}'
    elif col == 'abandono':
        tipo = 'TARGET'
        tipo_color = '#e53e3e'
        detalle = f'0 (no): {int((df[col]==0).sum()):,} | 1 (si): {int((df[col]==1).sum()):,}'.replace(',','.')
    else:
        tipo = 'Categorica'
        tipo_color = '#38a169'
        top2 = df[col].value_counts().head(2)
        detalle = ' | '.join([f'{v[:18]}: {c:,}'.replace(',','.') for v, c in top2.items()])

    nulo_badge = ''
    if pct_nulos > 0:
        nulo_color = '#e53e3e' if pct_nulos > 30 else '#ed8936' if pct_nulos > 10 else '#ecc94b'
        nulo_badge = f'<span style="background:{nulo_color};color:white;padding:1px 6px;border-radius:4px;font-size:0.75em;margin-left:5px;">{pct_nulos:.1f}% nulos</span>'

    fichas_html += (
        f'<div style="background:#f7fafc;border-radius:8px;padding:10px;border-left:3px solid {tipo_color};">'
        f'<div style="font-weight:bold;font-size:0.9em;">{nombre}{nulo_badge}</div>'
        f'<div style="font-size:0.75em;color:{tipo_color};margin:2px 0;">{tipo} | {n_unicos} unicos</div>'
        f'<div style="font-size:0.75em;color:#718096;">{detalle}</div>'
        f'</div>'
    )
fichas_html += '</div>'

s_vars = generar_seccion_html('Catalogo de Variables', fichas_html)

# DataTables: CSS y JS para tablas interactivas
datatables_cdn = (
    '<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">'
    '<script src="https://code.jquery.com/jquery-3.7.0.min.js"></script>'
    '<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>'
)

datatables_init = (
    '<script>'
    '$(document).ready(function(){'
    '$(".tabla-dt").DataTable({'
    'language:{'
    'search:"Buscar:",'
    'lengthMenu:"Mostrar _MENU_ filas",'
    'info:"Mostrando _START_ a _END_ de _TOTAL_",'
    'paginate:{first:"Primero",last:"Ultimo",next:"Sig.",previous:"Ant."},'
    'zeroRecords:"Sin resultados",'
    'infoEmpty:"Sin datos",'
    'infoFiltered:"(filtrado de _MAX_ totales)"'
    '},'
    'pageLength:5,'
    'order:[],'
    'dom:"<\'top\'><\'clear\'>rt<\'bottom\'ip>"'
    '});'
    '});'
    '</script>'
    '<style>'
    '.dataTables_wrapper{font-size:0.8em;margin:0;padding:0;}'
    '.dataTables_filter input{border:1px solid #cbd5e0;border-radius:6px;padding:4px 8px;}'
    'table.tabla-dt{border-collapse:collapse;width:100%;}'
    'table.tabla-dt thead th{background:#3182ce;color:white;padding:4px 6px;text-align:left;cursor:pointer;font-size:0.9em;}'
    'table.tabla-dt tbody td{padding:3px 6px;border-bottom:1px solid #e2e8f0;}'
    'table.tabla-dt tbody tr:hover{background:#ebf8ff;}'
    'table.tabla-dt tbody tr:nth-child(even){background:#f7fafc;}'
    '.dataTables_info,.dataTables_paginate{margin-top:8px;font-size:0.85em;}'
    '.dataTables_paginate .paginate_button{padding:3px 8px;margin:0 2px;border:1px solid #cbd5e0;border-radius:4px;cursor:pointer;}'
    '.dataTables_paginate .paginate_button.current{background:#3182ce;color:white;border-color:#3182ce;}'
    '</style>'
)



# --- Diagrama PRISMA (SVG externo) ---
prisma_path = ROOT / 'docs' / 'img' / 'prisma_flujo_datos.svg'
if prisma_path.exists():
    prisma_svg = prisma_path.read_text(encoding='utf-8')
else:
    prisma_svg = '<p style="color:#e53e3e;">SVG no encontrado en docs/img/prisma_flujo_datos.svg</p>'

prisma = (
'<div style="margin-bottom:20px;">'
'<button onclick="document.getElementById(\'prisma-flow\').style.display='
'document.getElementById(\'prisma-flow\').style.display===\'none\'?\'block\':\'none\'" '
'style="background:#3182ce;color:white;border:none;padding:10px 20px;'
'border-radius:8px;cursor:pointer;font-size:1em;font-weight:bold;">'
'&#x1f4ca; Ver Diagrama de Flujo PRISMA</button>'
'<div id="prisma-flow" style="display:none;margin-top:15px;padding:20px;'
'background:white;border-radius:12px;border:1px solid #e2e8f0;">'
f'{prisma_svg}'
'<p style="text-align:center;font-size:0.8em;color:#718096;margin-top:5px;">'
'Flujo de seleccion de datos desde fuentes originales hasta dataset final.</p>'
'</div></div>'
)

# --- Modelo conceptual del estudiante (SVG externo) ---
svg_path = ROOT / 'docs' / 'img' / 'modelo_conceptual_estudiante.svg'
if svg_path.exists():
    svg_content = svg_path.read_text(encoding='utf-8')
else:
    svg_content = '<p style="color:#e53e3e;">SVG no encontrado en docs/img/modelo_conceptual_estudiante.svg</p>'

modelo = (
'<div style="margin-bottom:20px;">'
'<button onclick="document.getElementById(\'modelo-est\').style.display='
'document.getElementById(\'modelo-est\').style.display===\'none\'?\'block\':\'none\'" '
'style="background:#805ad5;color:white;border:none;padding:10px 20px;'
'border-radius:8px;cursor:pointer;font-size:1em;font-weight:bold;margin-left:10px;">'
'&#x1f393; Ver Modelo Conceptual del Estudiante</button>'
'<div id="modelo-est" style="display:none;margin-top:15px;padding:20px;'
'background:white;border-radius:12px;border:1px solid #e2e8f0;">'
f'{svg_content}'
'<p style="text-align:center;font-size:0.8em;color:#718096;margin-top:5px;">'
'Adaptado de McNeill (2010). 20 variables del dataset mapeadas a 4 zonas de interaccion.</p>'
'</div></div>'
)

contenido = (
    prisma
    + modelo
    + generar_kpis_html(KPIS)
    + s_target
    + s_nulos
    + s_desc
    + s_outliers
    + s_vars

)


html = render_base_html(
    titulo='M01: Inspección del Dataset',
    icono=chr(0x1f50d),
    subtitulo='Fase 4: EDA Final | TFM Abandono Universitario',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre='f4_m01_inspeccion.ipynb',
    notebook_carpeta='fase4_eda'
)

ruta_html = RUTA_FASE4_HTML / 'm01_inspeccion.html'
guardar_html(html, ruta_html)
print(f'HTML generado: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m01_inspeccion.html
HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m01_inspeccion.html


In [9]:
# ============================================================================
# CELDA 8: RESUMEN
# ============================================================================

print('='*70)
print('F4-M01: INSPECCION - RESUMEN')
print('='*70)
print(f'Dataset:       df_eda_final.parquet')
print(f'Filas:         {fmt(n_filas)}')
print(f'Columnas:      {n_cols}')
print(f'Numericas:     {len(cols_num_cont)} continuas + {len(cols_bin)} binarias')
print(f'Categoricas:   {len(cols_cat)}')
print(f'Abandono:      {pct_abandono:.1f}%')
print(f'Vars con nulos:{n_vars_nulas}')
print(f'Graficos:      {len(graficos_base64)}')
print(f'HTML:          {ruta_html}')
print()
print('Siguiente: f4_m02_target.ipynb')

F4-M01: INSPECCION - RESUMEN
Dataset:       df_eda_final.parquet
Filas:         33.621
Columnas:      21
Numericas:     10 continuas + 2 binarias
Categoricas:   8
Abandono:      29.2%
Vars con nulos:7
Graficos:      5
HTML:          C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m01_inspeccion.html

Siguiente: f4_m02_target.ipynb
